# Transformation - Flight data

### Transformation Checklist
1. [x] Remove unnecessary time in `fl_date` column and make sure its `dd/mm/yyyy`.
2. [x] Drop unncessary columns
   1. `op_carrier_airline_id` - Redundant
   2. `tail_num` - Unnecessary
   3. `origin_airport_seq_id` - Redandant
   4. `origin_city_market_id` - Unnecessary
   5. `origin_state_fips` - Unnecessary
   6. `origin_state_nm` - Redandant
   7. `origin_wac` - Unnecessary
   8. `dest_airport_seq_id` - Redundant
   9. `dest_state_nm` - Redundant
   10. `dest_wac` - Unnecessary
   11. `cancelled` - Cancellation status is not relevant for the analysis
   12. `cancellation_code` - Unnecessary
   13. `diverted` - Diversion status is not relevant for the analysis
   14. `actual_elapsed_time` - Flight time is not relevant for this analysis
   15. `flights` - Extreamly low variance column, only single value accoross the entire dataset
   16. `FIRST_DEP_TIME` - Unnecessary
   17. `TOTAL_ADD_GTIME` - Unnecessary
3. [x] Type correction across all columns
4. [x] Deal with missing values
5. [x] Standardize all the date and time fields to use the same format 

In [1]:
import sys
from pathlib import Path

project_root = "/notebooks/ML/Flight Delay Prediction Project/ELT"
sys.path.append(str(project_root))

In [2]:
from pyspark.sql.types import *
from pyspark.sql.functions import *
import pandas as pd

In [3]:
from src.utils.spark_session import get_spark_session
from src.transform.flight_data import *
from src.load.flight_data import *

In [4]:
pd.set_option('display.max_columns', None)

In [ ]:
spark = get_spark_session()

In [6]:
s3_key = "bronze/Flight_Delay_Prediction_Datasets/flight_data/"
spark_df = get_lake_dataset(spark, s3_key)

SLF4J: Failed to load class "org.slf4j.impl.StaticLoggerBinder".
SLF4J: Defaulting to no-operation (NOP) logger implementation
SLF4J: See http://www.slf4j.org/codes.html#StaticLoggerBinder for further details.
                                                                                

In [7]:
spark_df.limit(5).toPandas()

,YEAR,QUARTER,MONTH,DAY_OF_MONTH,DAY_OF_WEEK,FL_DATE,OP_UNIQUE_CARRIER,OP_CARRIER_AIRLINE_ID,OP_CARRIER,TAIL_NUM,ORIGIN_AIRPORT_ID,ORIGIN_AIRPORT_SEQ_ID,ORIGIN_CITY_MARKET_ID,ORIGIN,ORIGIN_CITY_NAME,ORIGIN_STATE_ABR,ORIGIN_STATE_FIPS,ORIGIN_STATE_NM,ORIGIN_WAC,DEST_AIRPORT_ID,DEST_AIRPORT_SEQ_ID,DEST,DEST_CITY_NAME,DEST_STATE_ABR,DEST_STATE_NM,DEST_WAC,CRS_DEP_TIME,DEP_TIME,DEP_DELAY,DEP_DELAY_NEW,DEP_DEL15,DEP_DELAY_GROUP,DEP_TIME_BLK,TAXI_OUT,WHEELS_OFF,WHEELS_ON,TAXI_IN,CRS_ARR_TIME,ARR_TIME,ARR_DELAY,ARR_DELAY_NEW,ARR_DEL15,ARR_DELAY_GROUP,ARR_TIME_BLK,CANCELLED,CANCELLATION_CODE,DIVERTED,CRS_ELAPSED_TIME,ACTUAL_ELAPSED_TIME,AIR_TIME,FLIGHTS,DISTANCE,DISTANCE_GROUP,CARRIER_DELAY,WEATHER_DELAY,NAS_DELAY,SECURITY_DELAY,LATE_AIRCRAFT_DELAY,FIRST_DEP_TIME,TOTAL_ADD_GTIME
0,2025,2,5,16,5,5/16/2025 12:00:00 AM,WN,19393,WN,N8317M,13796,1379610,32457,OAK,"Oakland, CA",CA,6,California,91,11292,1129202,DEN,"Denver, CO",CO,Colorado,82,1510,1517,7.0,7.0,0.0,0,1500-1559,8.0,1525,1823,7.0,1840,1830,-10.0,0.0,0.0,-1,1800-1859,0.0,None,0.0,150.0,133.0,118.0,1.0,957.0,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2025,2,5,16,5,5/16/2025 12:00:00 AM,WN,19393,WN,N8317M,14893,1489302,33192,SMF,"Sacramento, CA",CA,6,California,91,12954,1295407,LGB,"Long Beach, CA",CA,California,91,1055,1052,-3.0,0.0,0.0,-1,1000-1059,10.0,1102,1206,4.0,1220,1210,-10.0,0.0,0.0,-1,1200-1259,0.0,None,0.0,85.0,78.0,64.0,1.0,387.0,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2025,2,5,16,5,5/16/2025 12:00:00 AM,WN,19393,WN,N8317M,15304,1530402,33195,TPA,"Tampa, FL",FL,12,Florida,33,12889,1288904,LAS,"Las Vegas, NV",NV,Nevada,85,550,600,10.0,10.0,0.0,0,0001-0559,10.0,610,752,4.0,750,756,6.0,6.0,0.0,0,0700-0759,0.0,None,0.0,300.0,296.0,282.0,1.0,1984.0,8,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2025,2,5,16,5,5/16/2025 12:00:00 AM,WN,19393,WN,N8318F,10693,1069302,30693,BNA,"Nashville, TN",TN,47,Tennessee,54,14193,1419306,PNS,"Pensacola, FL",FL,Florida,33,2045,2250,125.0,125.0,1.0,8,2000-2059,13.0,2303,2358,5.0,2205,3,118.0,118.0,1.0,7,2200-2259,0.0,None,0.0,80.0,73.0,55.0,1.0,391.0,2,0.0,0.0,0.0,0.0,118.0,NaN,NaN
4,2025,2,5,16,5,5/16/2025 12:00:00 AM,WN,19393,WN,N8318F,11042,1104205,30647,CLE,"Cleveland, OH",OH,39,Ohio,44,15016,1501606,STL,"St. Louis, MO",MO,Missouri,64,815,814,-1.0,0.0,0.0,-1,0800-0859,7.0,821,839,6.0,850,845,-5.0,0.0,0.0,-1,0800-0859,0.0,None,0.0,95.0,91.0,78.0,1.0,487.0,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Remove unnecessary time in `fl_date` column and make sure its `dd/mm/yyyy`

In [8]:
spark_df_t1 = parse_fl_date(spark_df, 'FL_DATE')

In [9]:
spark_df_t1.limit(5).toPandas()

,YEAR,QUARTER,MONTH,DAY_OF_MONTH,DAY_OF_WEEK,FL_DATE,OP_UNIQUE_CARRIER,OP_CARRIER_AIRLINE_ID,OP_CARRIER,TAIL_NUM,ORIGIN_AIRPORT_ID,ORIGIN_AIRPORT_SEQ_ID,ORIGIN_CITY_MARKET_ID,ORIGIN,ORIGIN_CITY_NAME,ORIGIN_STATE_ABR,ORIGIN_STATE_FIPS,ORIGIN_STATE_NM,ORIGIN_WAC,DEST_AIRPORT_ID,DEST_AIRPORT_SEQ_ID,DEST,DEST_CITY_NAME,DEST_STATE_ABR,DEST_STATE_NM,DEST_WAC,CRS_DEP_TIME,DEP_TIME,DEP_DELAY,DEP_DELAY_NEW,DEP_DEL15,DEP_DELAY_GROUP,DEP_TIME_BLK,TAXI_OUT,WHEELS_OFF,WHEELS_ON,TAXI_IN,CRS_ARR_TIME,ARR_TIME,ARR_DELAY,ARR_DELAY_NEW,ARR_DEL15,ARR_DELAY_GROUP,ARR_TIME_BLK,CANCELLED,CANCELLATION_CODE,DIVERTED,CRS_ELAPSED_TIME,ACTUAL_ELAPSED_TIME,AIR_TIME,FLIGHTS,DISTANCE,DISTANCE_GROUP,CARRIER_DELAY,WEATHER_DELAY,NAS_DELAY,SECURITY_DELAY,LATE_AIRCRAFT_DELAY,FIRST_DEP_TIME,TOTAL_ADD_GTIME
0,2025,2,5,16,5,2025-05-16,WN,19393,WN,N8317M,13796,1379610,32457,OAK,"Oakland, CA",CA,6,California,91,11292,1129202,DEN,"Denver, CO",CO,Colorado,82,1510,1517,7.0,7.0,0.0,0,1500-1559,8.0,1525,1823,7.0,1840,1830,-10.0,0.0,0.0,-1,1800-1859,0.0,None,0.0,150.0,133.0,118.0,1.0,957.0,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2025,2,5,16,5,2025-05-16,WN,19393,WN,N8317M,14893,1489302,33192,SMF,"Sacramento, CA",CA,6,California,91,12954,1295407,LGB,"Long Beach, CA",CA,California,91,1055,1052,-3.0,0.0,0.0,-1,1000-1059,10.0,1102,1206,4.0,1220,1210,-10.0,0.0,0.0,-1,1200-1259,0.0,None,0.0,85.0,78.0,64.0,1.0,387.0,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2025,2,5,16,5,2025-05-16,WN,19393,WN,N8317M,15304,1530402,33195,TPA,"Tampa, FL",FL,12,Florida,33,12889,1288904,LAS,"Las Vegas, NV",NV,Nevada,85,550,600,10.0,10.0,0.0,0,0001-0559,10.0,610,752,4.0,750,756,6.0,6.0,0.0,0,0700-0759,0.0,None,0.0,300.0,296.0,282.0,1.0,1984.0,8,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2025,2,5,16,5,2025-05-16,WN,19393,WN,N8318F,10693,1069302,30693,BNA,"Nashville, TN",TN,47,Tennessee,54,14193,1419306,PNS,"Pensacola, FL",FL,Florida,33,2045,2250,125.0,125.0,1.0,8,2000-2059,13.0,2303,2358,5.0,2205,3,118.0,118.0,1.0,7,2200-2259,0.0,None,0.0,80.0,73.0,55.0,1.0,391.0,2,0.0,0.0,0.0,0.0,118.0,NaN,NaN
4,2025,2,5,16,5,2025-05-16,WN,19393,WN,N8318F,11042,1104205,30647,CLE,"Cleveland, OH",OH,39,Ohio,44,15016,1501606,STL,"St. Louis, MO",MO,Missouri,64,815,814,-1.0,0.0,0.0,-1,0800-0859,7.0,821,839,6.0,850,845,-5.0,0.0,0.0,-1,0800-0859,0.0,None,0.0,95.0,91.0,78.0,1.0,487.0,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Drop unncessary columns

In [10]:
column_list = [
    'op_carrier_airline_id',
    'tail_num',
    'origin_airport_seq_id',
    'origin_city_market_id',
    'origin_state_fips'
    'origin_state_nm',
    'origin_wac',
    'dest_airport_seq_id',
    'dest_state_nm',
    'dest_wac',
    'cancelled',
    'cancellation_code',
    'diverted',
    'actual_elapsed_time',
    'flights',
    'FIRST_DEP_TIME',
    'TOTAL_ADD_GTIME'
]

In [11]:
spark_df_t2 = drop_columns(spark_df_t1, column_list)

In [12]:
spark_df_t2.limit(5).toPandas()

,YEAR,QUARTER,MONTH,DAY_OF_MONTH,DAY_OF_WEEK,FL_DATE,OP_UNIQUE_CARRIER,OP_CARRIER,ORIGIN_AIRPORT_ID,ORIGIN,ORIGIN_CITY_NAME,ORIGIN_STATE_ABR,ORIGIN_STATE_FIPS,ORIGIN_STATE_NM,DEST_AIRPORT_ID,DEST,DEST_CITY_NAME,DEST_STATE_ABR,CRS_DEP_TIME,DEP_TIME,DEP_DELAY,DEP_DELAY_NEW,DEP_DEL15,DEP_DELAY_GROUP,DEP_TIME_BLK,TAXI_OUT,WHEELS_OFF,WHEELS_ON,TAXI_IN,CRS_ARR_TIME,ARR_TIME,ARR_DELAY,ARR_DELAY_NEW,ARR_DEL15,ARR_DELAY_GROUP,ARR_TIME_BLK,CRS_ELAPSED_TIME,AIR_TIME,DISTANCE,DISTANCE_GROUP,CARRIER_DELAY,WEATHER_DELAY,NAS_DELAY,SECURITY_DELAY,LATE_AIRCRAFT_DELAY
0,2025,2,5,16,5,2025-05-16,WN,WN,13796,OAK,"Oakland, CA",CA,6,California,11292,DEN,"Denver, CO",CO,1510,1517,7.0,7.0,0.0,0,1500-1559,8.0,1525,1823,7.0,1840,1830,-10.0,0.0,0.0,-1,1800-1859,150.0,118.0,957.0,4,NaN,NaN,NaN,NaN,NaN
1,2025,2,5,16,5,2025-05-16,WN,WN,14893,SMF,"Sacramento, CA",CA,6,California,12954,LGB,"Long Beach, CA",CA,1055,1052,-3.0,0.0,0.0,-1,1000-1059,10.0,1102,1206,4.0,1220,1210,-10.0,0.0,0.0,-1,1200-1259,85.0,64.0,387.0,2,NaN,NaN,NaN,NaN,NaN
2,2025,2,5,16,5,2025-05-16,WN,WN,15304,TPA,"Tampa, FL",FL,12,Florida,12889,LAS,"Las Vegas, NV",NV,550,600,10.0,10.0,0.0,0,0001-0559,10.0,610,752,4.0,750,756,6.0,6.0,0.0,0,0700-0759,300.0,282.0,1984.0,8,NaN,NaN,NaN,NaN,NaN
3,2025,2,5,16,5,2025-05-16,WN,WN,10693,BNA,"Nashville, TN",TN,47,Tennessee,14193,PNS,"Pensacola, FL",FL,2045,2250,125.0,125.0,1.0,8,2000-2059,13.0,2303,2358,5.0,2205,3,118.0,118.0,1.0,7,2200-2259,80.0,55.0,391.0,2,0.0,0.0,0.0,0.0,118.0
4,2025,2,5,16,5,2025-05-16,WN,WN,11042,CLE,"Cleveland, OH",OH,39,Ohio,15016,STL,"St. Louis, MO",MO,815,814,-1.0,0.0,0.0,-1,0800-0859,7.0,821,839,6.0,850,845,-5.0,0.0,0.0,-1,0800-0859,95.0,78.0,487.0,2,NaN,NaN,NaN,NaN,NaN


In [13]:
print((spark_df_t2.count(), len(spark_df_t2.columns)))

[Stage 4:=======================================>                   (2 + 1) / 3]

(7001619, 45)


## Type correction

In [14]:
spark_df_t2.printSchema()

root
 |-- YEAR: integer (nullable = true)
 |-- QUARTER: integer (nullable = true)
 |-- MONTH: integer (nullable = true)
 |-- DAY_OF_MONTH: integer (nullable = true)
 |-- DAY_OF_WEEK: integer (nullable = true)
 |-- FL_DATE: date (nullable = true)
 |-- OP_UNIQUE_CARRIER: string (nullable = true)
 |-- OP_CARRIER: string (nullable = true)
 |-- ORIGIN_AIRPORT_ID: integer (nullable = true)
 |-- ORIGIN: string (nullable = true)
 |-- ORIGIN_CITY_NAME: string (nullable = true)
 |-- ORIGIN_STATE_ABR: string (nullable = true)
 |-- ORIGIN_STATE_FIPS: integer (nullable = true)
 |-- ORIGIN_STATE_NM: string (nullable = true)
 |-- DEST_AIRPORT_ID: integer (nullable = true)
 |-- DEST: string (nullable = true)
 |-- DEST_CITY_NAME: string (nullable = true)
 |-- DEST_STATE_ABR: string (nullable = true)
 |-- CRS_DEP_TIME: integer (nullable = true)
 |-- DEP_TIME: integer (nullable = true)
 |-- DEP_DELAY: double (nullable = true)
 |-- DEP_DELAY_NEW: double (nullable = true)
 |-- DEP_DEL15: double (nullable =

### Time type correction

In [15]:
time_columns = [
    'CRS_DEP_TIME',
    'DEP_TIME',
    'WHEELS_OFF',
    'WHEELS_ON',
    'CRS_ARR_TIME',
    'ARR_TIME'
]

In [16]:
spark_df_t3 = correct_hhmm(spark_df_t2, time_columns)

In [17]:
spark_df_t3.printSchema()

root
 |-- YEAR: integer (nullable = true)
 |-- QUARTER: integer (nullable = true)
 |-- MONTH: integer (nullable = true)
 |-- DAY_OF_MONTH: integer (nullable = true)
 |-- DAY_OF_WEEK: integer (nullable = true)
 |-- FL_DATE: date (nullable = true)
 |-- OP_UNIQUE_CARRIER: string (nullable = true)
 |-- OP_CARRIER: string (nullable = true)
 |-- ORIGIN_AIRPORT_ID: integer (nullable = true)
 |-- ORIGIN: string (nullable = true)
 |-- ORIGIN_CITY_NAME: string (nullable = true)
 |-- ORIGIN_STATE_ABR: string (nullable = true)
 |-- ORIGIN_STATE_FIPS: integer (nullable = true)
 |-- ORIGIN_STATE_NM: string (nullable = true)
 |-- DEST_AIRPORT_ID: integer (nullable = true)
 |-- DEST: string (nullable = true)
 |-- DEST_CITY_NAME: string (nullable = true)
 |-- DEST_STATE_ABR: string (nullable = true)
 |-- CRS_DEP_TIME: string (nullable = true)
 |-- DEP_TIME: string (nullable = true)
 |-- DEP_DELAY: double (nullable = true)
 |-- DEP_DELAY_NEW: double (nullable = true)
 |-- DEP_DEL15: double (nullable = t

In [18]:
spark_df_t3.limit(5).toPandas()

,YEAR,QUARTER,MONTH,DAY_OF_MONTH,DAY_OF_WEEK,FL_DATE,OP_UNIQUE_CARRIER,OP_CARRIER,ORIGIN_AIRPORT_ID,ORIGIN,ORIGIN_CITY_NAME,ORIGIN_STATE_ABR,ORIGIN_STATE_FIPS,ORIGIN_STATE_NM,DEST_AIRPORT_ID,DEST,DEST_CITY_NAME,DEST_STATE_ABR,CRS_DEP_TIME,DEP_TIME,DEP_DELAY,DEP_DELAY_NEW,DEP_DEL15,DEP_DELAY_GROUP,DEP_TIME_BLK,TAXI_OUT,WHEELS_OFF,WHEELS_ON,TAXI_IN,CRS_ARR_TIME,ARR_TIME,ARR_DELAY,ARR_DELAY_NEW,ARR_DEL15,ARR_DELAY_GROUP,ARR_TIME_BLK,CRS_ELAPSED_TIME,AIR_TIME,DISTANCE,DISTANCE_GROUP,CARRIER_DELAY,WEATHER_DELAY,NAS_DELAY,SECURITY_DELAY,LATE_AIRCRAFT_DELAY
0,2025,2,5,16,5,2025-05-16,WN,WN,13796,OAK,"Oakland, CA",CA,6,California,11292,DEN,"Denver, CO",CO,15:10,15:17,7.0,7.0,0.0,0,1500-1559,8.0,15:25,18:23,7.0,18:40,18:30,-10.0,0.0,0.0,-1,1800-1859,150.0,118.0,957.0,4,NaN,NaN,NaN,NaN,NaN
1,2025,2,5,16,5,2025-05-16,WN,WN,14893,SMF,"Sacramento, CA",CA,6,California,12954,LGB,"Long Beach, CA",CA,10:55,10:52,-3.0,0.0,0.0,-1,1000-1059,10.0,11:02,12:06,4.0,12:20,12:10,-10.0,0.0,0.0,-1,1200-1259,85.0,64.0,387.0,2,NaN,NaN,NaN,NaN,NaN
2,2025,2,5,16,5,2025-05-16,WN,WN,15304,TPA,"Tampa, FL",FL,12,Florida,12889,LAS,"Las Vegas, NV",NV,05:50,06:00,10.0,10.0,0.0,0,0001-0559,10.0,06:10,07:52,4.0,07:50,07:56,6.0,6.0,0.0,0,0700-0759,300.0,282.0,1984.0,8,NaN,NaN,NaN,NaN,NaN
3,2025,2,5,16,5,2025-05-16,WN,WN,10693,BNA,"Nashville, TN",TN,47,Tennessee,14193,PNS,"Pensacola, FL",FL,20:45,22:50,125.0,125.0,1.0,8,2000-2059,13.0,23:03,23:58,5.0,22:05,00:03,118.0,118.0,1.0,7,2200-2259,80.0,55.0,391.0,2,0.0,0.0,0.0,0.0,118.0
4,2025,2,5,16,5,2025-05-16,WN,WN,11042,CLE,"Cleveland, OH",OH,39,Ohio,15016,STL,"St. Louis, MO",MO,08:15,08:14,-1.0,0.0,0.0,-1,0800-0859,7.0,08:21,08:39,6.0,08:50,08:45,-5.0,0.0,0.0,-1,0800-0859,95.0,78.0,487.0,2,NaN,NaN,NaN,NaN,NaN


## Missing values

In [19]:
import pyspark.sql.functions as F

In [20]:
spark_df_t3.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in spark_df_t3.columns
]).toPandas()

,YEAR,QUARTER,MONTH,DAY_OF_MONTH,DAY_OF_WEEK,FL_DATE,OP_UNIQUE_CARRIER,OP_CARRIER,ORIGIN_AIRPORT_ID,ORIGIN,ORIGIN_CITY_NAME,ORIGIN_STATE_ABR,ORIGIN_STATE_FIPS,ORIGIN_STATE_NM,DEST_AIRPORT_ID,DEST,DEST_CITY_NAME,DEST_STATE_ABR,CRS_DEP_TIME,DEP_TIME,DEP_DELAY,DEP_DELAY_NEW,DEP_DEL15,DEP_DELAY_GROUP,DEP_TIME_BLK,TAXI_OUT,WHEELS_OFF,WHEELS_ON,TAXI_IN,CRS_ARR_TIME,ARR_TIME,ARR_DELAY,ARR_DELAY_NEW,ARR_DEL15,ARR_DELAY_GROUP,ARR_TIME_BLK,CRS_ELAPSED_TIME,AIR_TIME,DISTANCE,DISTANCE_GROUP,CARRIER_DELAY,WEATHER_DELAY,NAS_DELAY,SECURITY_DELAY,LATE_AIRCRAFT_DELAY
0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,98170,98591,98591,98591,98591,0,102183,102183,104608,104608,0,104608,122135,122135,122135,122135,0,2,122135,0,0,5466981,5466981,5466981,5466981,5466981


In [21]:
spark_df_t3.select(F.count(F.when(F.col('DEP_TIME').isNull(), 'DEP_TIME')).alias('DEP_TIME')).first()[0]

98170

In [22]:
def calc_missing_percent(spark_df):
    missing_col_val = []
    
    for c in spark_df.columns:
        missing_cnt = spark_df.select(F.count(F.when(F.col(c).isNull(), c)).alias(c)).first()[0]
    
        if missing_cnt != 0:
            cnt_all = spark_df.select(F.count('*')).first()[0]
            missing_percent = missing_cnt/cnt_all * 100
            missing_col_val.append({"column": c, "missing_percent": missing_percent})

    for mcol in missing_col_val:
        print(f"Column:{mcol['column']}\t\t|\tMissing percentage: {mcol['missing_percent']}%")

In [23]:
calc_missing_percent(spark_df_t3)

Column:DEP_TIME		|	Missing percentage: 1.4021042847375729%
Column:DEP_DELAY		|	Missing percentage: 1.4081171797551395%
Column:DEP_DELAY_NEW		|	Missing percentage: 1.4081171797551395%
Column:DEP_DEL15		|	Missing percentage: 1.4081171797551395%
Column:DEP_DELAY_GROUP		|	Missing percentage: 1.4081171797551395%
Column:TAXI_OUT		|	Missing percentage: 1.4594195999525252%
Column:WHEELS_OFF		|	Missing percentage: 1.4594195999525252%
Column:WHEELS_ON		|	Missing percentage: 1.494054446550148%
Column:TAXI_IN		|	Missing percentage: 1.494054446550148%
Column:ARR_TIME		|	Missing percentage: 1.494054446550148%
Column:ARR_DELAY		|	Missing percentage: 1.7443822635878932%
Column:ARR_DELAY_NEW		|	Missing percentage: 1.7443822635878932%
Column:ARR_DEL15		|	Missing percentage: 1.7443822635878932%
Column:ARR_DELAY_GROUP		|	Missing percentage: 1.7443822635878932%
Column:CRS_ELAPSED_TIME		|	Missing percentage: 2.8564821936183618e-05%
Column:AIR_TIME		|	Missing percentage: 1.7443822635878932%
Column:CARRIER_DE

In [24]:
fill_columns = ['CARRIER_DELAY', 'WEATHER_DELAY', 'NAS_DELAY', 'SECURITY_DELAY', 'LATE_AIRCRAFT_DELAY']

In [25]:
spark_df_t4 = fill_missing_values(spark_df_t3, fill_columns, 0)

In [26]:
spark_df_t5 = drop_missing_values(spark_df_t4)

In [27]:
print((spark_df_t5.count(), len(spark_df_t5.columns)))

[Stage 212:======================================>                  (2 + 1) / 3]

(6879484, 45)


In [29]:
spark_df_t5.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in spark_df_t5.columns
]).toPandas()

,YEAR,QUARTER,MONTH,DAY_OF_MONTH,DAY_OF_WEEK,FL_DATE,OP_UNIQUE_CARRIER,OP_CARRIER,ORIGIN_AIRPORT_ID,ORIGIN,ORIGIN_CITY_NAME,ORIGIN_STATE_ABR,ORIGIN_STATE_FIPS,ORIGIN_STATE_NM,DEST_AIRPORT_ID,DEST,DEST_CITY_NAME,DEST_STATE_ABR,CRS_DEP_TIME,DEP_TIME,DEP_DELAY,DEP_DELAY_NEW,DEP_DEL15,DEP_DELAY_GROUP,DEP_TIME_BLK,TAXI_OUT,WHEELS_OFF,WHEELS_ON,TAXI_IN,CRS_ARR_TIME,ARR_TIME,ARR_DELAY,ARR_DELAY_NEW,ARR_DEL15,ARR_DELAY_GROUP,ARR_TIME_BLK,CRS_ELAPSED_TIME,AIR_TIME,DISTANCE,DISTANCE_GROUP,CARRIER_DELAY,WEATHER_DELAY,NAS_DELAY,SECURITY_DELAY,LATE_AIRCRAFT_DELAY
0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


## Standadization

In [30]:
spark_df_t5.limit(5).toPandas()

,YEAR,QUARTER,MONTH,DAY_OF_MONTH,DAY_OF_WEEK,FL_DATE,OP_UNIQUE_CARRIER,OP_CARRIER,ORIGIN_AIRPORT_ID,ORIGIN,ORIGIN_CITY_NAME,ORIGIN_STATE_ABR,ORIGIN_STATE_FIPS,ORIGIN_STATE_NM,DEST_AIRPORT_ID,DEST,DEST_CITY_NAME,DEST_STATE_ABR,CRS_DEP_TIME,DEP_TIME,DEP_DELAY,DEP_DELAY_NEW,DEP_DEL15,DEP_DELAY_GROUP,DEP_TIME_BLK,TAXI_OUT,WHEELS_OFF,WHEELS_ON,TAXI_IN,CRS_ARR_TIME,ARR_TIME,ARR_DELAY,ARR_DELAY_NEW,ARR_DEL15,ARR_DELAY_GROUP,ARR_TIME_BLK,CRS_ELAPSED_TIME,AIR_TIME,DISTANCE,DISTANCE_GROUP,CARRIER_DELAY,WEATHER_DELAY,NAS_DELAY,SECURITY_DELAY,LATE_AIRCRAFT_DELAY
0,2025,2,5,16,5,2025-05-16,WN,WN,13796,OAK,"Oakland, CA",CA,6,California,11292,DEN,"Denver, CO",CO,15:10,15:17,7.0,7.0,0.0,0,1500-1559,8.0,15:25,18:23,7.0,18:40,18:30,-10.0,0.0,0.0,-1,1800-1859,150.0,118.0,957.0,4,0.0,0.0,0.0,0.0,0.0
1,2025,2,5,16,5,2025-05-16,WN,WN,14893,SMF,"Sacramento, CA",CA,6,California,12954,LGB,"Long Beach, CA",CA,10:55,10:52,-3.0,0.0,0.0,-1,1000-1059,10.0,11:02,12:06,4.0,12:20,12:10,-10.0,0.0,0.0,-1,1200-1259,85.0,64.0,387.0,2,0.0,0.0,0.0,0.0,0.0
2,2025,2,5,16,5,2025-05-16,WN,WN,15304,TPA,"Tampa, FL",FL,12,Florida,12889,LAS,"Las Vegas, NV",NV,05:50,06:00,10.0,10.0,0.0,0,0001-0559,10.0,06:10,07:52,4.0,07:50,07:56,6.0,6.0,0.0,0,0700-0759,300.0,282.0,1984.0,8,0.0,0.0,0.0,0.0,0.0
3,2025,2,5,16,5,2025-05-16,WN,WN,10693,BNA,"Nashville, TN",TN,47,Tennessee,14193,PNS,"Pensacola, FL",FL,20:45,22:50,125.0,125.0,1.0,8,2000-2059,13.0,23:03,23:58,5.0,22:05,00:03,118.0,118.0,1.0,7,2200-2259,80.0,55.0,391.0,2,0.0,0.0,0.0,0.0,118.0
4,2025,2,5,16,5,2025-05-16,WN,WN,11042,CLE,"Cleveland, OH",OH,39,Ohio,15016,STL,"St. Louis, MO",MO,08:15,08:14,-1.0,0.0,0.0,-1,0800-0859,7.0,08:21,08:39,6.0,08:50,08:45,-5.0,0.0,0.0,-1,0800-0859,95.0,78.0,487.0,2,0.0,0.0,0.0,0.0,0.0


* All date and time columns uses the same format, so no standadization needed.

## Export the dataset and store in the data lake house

In [ ]:
s3_key = "cleaned/Flight_Delay_Prediction_Datasets/flight_data/"
write_transformed_data(spark, spark_df_t5, s3_key)